In [1]:
import os
import glob
import numpy as np
import xarray as xr
from tqdm import tqdm

# =========================
# 1. 路径与参数
# =========================
input_dir = r"E:\xuanxuan\changjiang_variables_era5_global"
output_dir = r"E:\xuanxuan\changjiang_variables_era5_global_mask"
ref_file = os.path.join(input_dir, "Volumetric-soil-water-layer-1-pentad_1982_2022.nc")

# 海洋阈值：
# 若某格点在 swvl1 中“所有时刻 abs(value) <= THRESHOLD”，则认为是海洋格点
THRESHOLD = 1e-4   # 你也可以改成 1e-3

os.makedirs(output_dir, exist_ok=True)


# =========================
# 2. 工具函数
# =========================
def find_main_var(ds):
    """
    找到数据集中的唯一主变量
    假设每个nc只有唯一数据变量
    """
    data_vars = list(ds.data_vars)
    if len(data_vars) != 1:
        raise ValueError(f"数据变量数量不是1，而是 {len(data_vars)} 个：{data_vars}")
    return data_vars[0]


def find_dim_name(da, candidates, dim_type="unknown"):
    """
    在 DataArray 的 dims 中寻找指定类型的维度名
    """
    dims_lower_map = {d.lower(): d for d in da.dims}
    for c in candidates:
        if c.lower() in dims_lower_map:
            return dims_lower_map[c.lower()]

    raise ValueError(f"未能识别 {dim_type} 维度，当前维度为：{da.dims}")


def get_time_lat_lon_names(da):
    """
    自动识别 time / lat / lon 维度名
    """
    time_name = find_dim_name(
        da,
        candidates=["time", "valid_time"],
        dim_type="time"
    )
    lat_name = find_dim_name(
        da,
        candidates=["latitude", "lat"],
        dim_type="latitude"
    )
    lon_name = find_dim_name(
        da,
        candidates=["longitude", "lon"],
        dim_type="longitude"
    )
    return time_name, lat_name, lon_name


def align_mask_to_target(mask_2d_ref, target_lat, target_lon):
    """
    将参考海洋mask按目标文件的lat/lon顺序对齐
    要求网格大小一致，且坐标要么完全一致，要么仅仅是反向排列
    """
    ref_lat = mask_2d_ref.coords[mask_2d_ref.dims[0]].values
    ref_lon = mask_2d_ref.coords[mask_2d_ref.dims[1]].values

    tgt_lat = target_lat.values
    tgt_lon = target_lon.values

    # 检查纬度
    if len(ref_lat) != len(tgt_lat):
        raise ValueError(f"纬度长度不一致：ref={len(ref_lat)}, target={len(tgt_lat)}")
    if np.allclose(ref_lat, tgt_lat, equal_nan=True):
        lat_slice = slice(None)
    elif np.allclose(ref_lat[::-1], tgt_lat, equal_nan=True):
        lat_slice = slice(None, None, -1)
    else:
        raise ValueError("纬度坐标不一致，且不是简单反向，无法对齐。")

    # 检查经度
    if len(ref_lon) != len(tgt_lon):
        raise ValueError(f"经度长度不一致：ref={len(ref_lon)}, target={len(tgt_lon)}")
    if np.allclose(ref_lon, tgt_lon, equal_nan=True):
        lon_slice = slice(None)
    elif np.allclose(ref_lon[::-1], tgt_lon, equal_nan=True):
        lon_slice = slice(None, None, -1)
    else:
        raise ValueError("经度坐标不一致，且不是简单反向，无法对齐。")

    aligned = mask_2d_ref.isel(
        {mask_2d_ref.dims[0]: lat_slice, mask_2d_ref.dims[1]: lon_slice}
    )

    return aligned


# =========================
# 3. 读取参考文件，构建海洋mask
# =========================
print(">>> 正在读取参考文件并构建海洋mask ...")
if not os.path.exists(ref_file):
    raise FileNotFoundError(f"参考文件不存在：{ref_file}")

with xr.open_dataset(ref_file) as ds_ref:
    ref_var_name = find_main_var(ds_ref)
    ref_da = ds_ref[ref_var_name]

    if ref_da.ndim != 3:
        raise ValueError(f"参考变量不是3维，而是 {ref_da.ndim} 维，变量名={ref_var_name}, dims={ref_da.dims}")

    ref_time_name, ref_lat_name, ref_lon_name = get_time_lat_lon_names(ref_da)

    # 转为统一顺序 (time, lat, lon)
    ref_da_std = ref_da.transpose(ref_time_name, ref_lat_name, ref_lon_name)

    # 载入内存
    ref_np = ref_da_std.values  # shape = (time, lat, lon)

    # 判定海洋格点：
    # 所有时刻均为有限值且 abs(value) <= THRESHOLD
    ocean_mask_np = np.all(np.isfinite(ref_np) & (np.abs(ref_np) <= THRESHOLD), axis=0)

    ocean_mask_ref = xr.DataArray(
        ocean_mask_np,
        dims=(ref_lat_name, ref_lon_name),
        coords={
            ref_lat_name: ref_da_std[ref_lat_name].values,
            ref_lon_name: ref_da_std[ref_lon_name].values
        },
        name="ocean_mask"
    )

print(f">>> 海洋mask构建完成：海洋格点数 = {int(ocean_mask_ref.values.sum())}")


# =========================
# 4. 遍历所有nc并处理
# =========================
nc_files = sorted(glob.glob(os.path.join(input_dir, "*.nc")))
print(f">>> 共发现 {len(nc_files)} 个 nc 文件")

if len(nc_files) == 0:
    raise FileNotFoundError(f"在目录中未找到 .nc 文件：{input_dir}")

for nc_path in tqdm(nc_files, desc="Masking nc files"):
    file_name = os.path.basename(nc_path)
    out_path = os.path.join(output_dir, file_name)

    try:
        with xr.open_dataset(nc_path) as ds:
            var_name = find_main_var(ds)
            da = ds[var_name]

            if da.ndim != 3:
                raise ValueError(f"{file_name} 中变量 {var_name} 不是3维，dims={da.dims}")

            time_name, lat_name, lon_name = get_time_lat_lon_names(da)

            # 对齐海洋mask到当前文件的lat/lon顺序
            mask_cur = align_mask_to_target(
                ocean_mask_ref,
                target_lat=da[lat_name],
                target_lon=da[lon_name]
            )

            # 将mask的维度名改成当前文件的维度名（如果不同）
            rename_dict = {}
            if mask_cur.dims[0] != lat_name:
                rename_dict[mask_cur.dims[0]] = lat_name
            if mask_cur.dims[1] != lon_name:
                rename_dict[mask_cur.dims[1]] = lon_name
            if len(rename_dict) > 0:
                mask_cur = mask_cur.rename(rename_dict)

            # 若原变量不是浮点型，需要先转浮点型才能写入 NaN
            original_dtype = da.dtype
            if not np.issubdtype(original_dtype, np.floating):
                da_masked = da.astype(np.float32)
            else:
                da_masked = da.copy()

            # 海洋格点统一赋值为 NaN
            # mask_cur == True 的地方为海洋
            da_masked = da_masked.where(~mask_cur, np.nan)

            # 保留原属性
            da_masked.attrs = da.attrs

            # 构建输出数据集，保持变量名、维度名、坐标名不变
            ds_out = ds.copy(deep=False)
            ds_out[var_name] = da_masked
            ds_out.attrs = ds.attrs

            # 编码：压缩保存
            encoding = {}
            encoding[var_name] = {
                "zlib": True,
                "complevel": 4
            }

            # 若原本有 _FillValue，可保留；否则让 xarray 自动处理
            if "_FillValue" in da.encoding:
                encoding[var_name]["_FillValue"] = da.encoding["_FillValue"]

            ds_out.to_netcdf(out_path, mode="w", format="NETCDF4", engine="netcdf4", encoding=encoding)

    except Exception as e:
        print(f"\n[跳过] 文件处理失败：{file_name}")
        print(f"原因：{repr(e)}")

print("\n>>> 全部处理完成。")
print(f">>> 输出目录：{output_dir}")

>>> 正在读取参考文件并构建海洋mask ...


F:\anaconda\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been deprecated and will be removed in a future release
  "class": algorithms.Blowfish,


>>> 海洋mask构建完成：海洋格点数 = 671799
>>> 共发现 41 个 nc 文件


Masking nc files: 100%|█████████████████████████████████████████████████████████████| 41/41 [2:48:26<00:00, 246.49s/it]


>>> 全部处理完成。
>>> 输出目录：E:\xuanxuan\changjiang_variables_era5_global_mask
